In [1]:
import time
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji, re, string

import nltk
from nltk.corpus import stopwords
import spacy

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

import warnings
warnings.filterwarnings("ignore")



d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../../Datasets/Telegram_Tratado_Rotulado_Revisado_Final.csv')
df.head()

,text_content_anonymous,preconceito
0,"Sim, eu sou um ""decepcionado""!\n\nEu vejo pess...",0
1,"Em épocas com muitos eventos grandes, altas qu...",0
2,Algumas premissas que ele adotou:\n- Ser de di...,0
3,Apocalipse 22:11. Cristo fez expiação por Seu ...,0
4,:sun_with_face:S:rose:H:cherry_blossom:A:sunfl...,0


In [3]:
df.shape

(3000, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   text_content_anonymous  3000 non-null   object
 1   preconceito             3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


In [5]:
mensagens_coluna = 'text_content_anonymous'  
df[mensagens_coluna] = df[mensagens_coluna].astype(str) # garantindo que todos os valores são strings

# Tokenizar mensagens (simples divisão por espaços)
df['num_tokens'] = df[mensagens_coluna].apply(lambda x: len(x.split()))

## Processamento do texto

In [6]:
unicode_emoji = {}
for key, value in emoji.EMOJI_DATA.items():
    try:
        unicode_emoji[key] = value['pt']
    except:
        pass

#emojis and punctuation
emojis_list = list(unicode_emoji)
punct = list(string.punctuation)
emojis_punct = emojis_list + punct

def processEmojisPunctuation(text, remove_punct = True):
    '''
    Put spaces between emojis. Removes punctuation.
    '''
    #get all unique chars
    chars = set(text)
    #for each unique char in text, do:
    for c in chars:
        #remove punctuation
        if remove_punct:
            if c in emojis_list:
                text = text.replace(c, ' ' + c + ' ')
            if c in punct:
                text = text.replace(c, ' ')

        #put spaces between punctuation
        else:
            if c in emojis_punct:
                text = text.replace(c, ' ' + c + ' ')          

    text = text.replace('  ', ' ')
    return text

#stop words removal
stop_words = list(stopwords.words('portuguese'))
new_stopwords = ['aí','pra','vão','vou','onde','lá','aqui',
                 'tá','pode','pois','so','deu','agora','todo',
                 'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq',
                 'cara','to','mim','la','vcs','tbm', 'tudo']
stop_words = stop_words + new_stopwords
final_stop_words = []
for sw in stop_words:
    sw = ' '+ sw + ' '
    final_stop_words.append(sw)

def removeStopwords(text):
    for sw in final_stop_words:
        text = text.replace(sw,' ')
    text = text.replace('  ',' ')
    return text

#lemmatization
nlp = spacy.load('pt_core_news_sm')
def lemmatization(text):
    doc = nlp(text)
    for token in doc:
        if token.text != token.lemma_:
            text = text.replace(token.text, token.lemma_)
    return text


def domainUrl(text):
    '''
    Substitutes an URL in a text for the domain of this URL
    Input: an string
    Output: the string with the modified URL
    '''    
    if 'http' in text:
        re_url = '[^\s]*https*://[^\s]*'
        matches = re.findall(re_url, text, flags=re.IGNORECASE)
        for m in matches:
            domain = m.split('//')
            domain = domain[1].split('/')[0]
            text = re.sub(re_url, domain, text, 1)
        return text
    else:
        return text 

def preprocess(text):
    text = text.lower().strip()
    text = domainUrl(text)
    text = processEmojisPunctuation(text)
    text = removeStopwords(text)
    text = lemmatization(text)
    return text


## Carregando dicionário 

In [7]:
# Função para carregar o dicionário
def carregar_dicionario_personalizado(dic_path):
    categorias = {}
    lexicon = {}
    dentro_das_categorias = False

    with open(dic_path, 'r', encoding='utf-8') as file:
        for linha in file:
            linha = linha.strip()
            
            # Detecta a seção de categorias delimitada por '%'
            if linha == '%':
                dentro_das_categorias = not dentro_das_categorias
                continue
            
            # Lê as categorias personalizadas
            if dentro_das_categorias:
                codigo, categoria = linha.split()
                categorias[codigo] = categoria
            else:
                # Lê as palavras e suas categorias
                partes = linha.split("\t")
                palavra = partes[0]
                categoria_ids = partes[1:]
                lexicon[palavra] = [categorias[codigo] for codigo in categoria_ids if codigo in categorias]
    
    return lexicon, list(categorias.values())

In [8]:
# Tokenização simples
def tokenize(text):
    tokens = []
    for match in re.finditer(r"\w+", text, re.UNICODE):
        tokens.append(match.group(0).lower())
    return tokens

# Função para incluir palavras do dicionário como colunas
def adicionar_palavras_como_features(df, lexicon):
    # Criar DataFrame temporário com todas as palavras inicializadas com 0
    new_columns = pd.DataFrame(0, index=df.index, columns=list(lexicon.keys()))
    
    # Concatenar com o DataFrame original de uma vez
    df = pd.concat([df, new_columns], axis=1)
    
    # Contar ocorrências de cada palavra no texto
    for i, texto in df['text'].items():
        tokens = tokenize(texto)
        for token in tokens:
            if token in lexicon:
                df.at[i, token] += 1
                
    return df

# Caminho do arquivo .dic personalizado
dic_path = '../../Dicionário/v2_SocialLIWC_formatado_ordenado.dic'
lexicon, category_names = carregar_dicionario_personalizado(dic_path)

In [9]:
# === Renomeando colunas ===
# Agora sim, renomeia para usar no modelo
df = df.rename(columns={
    "text_content_anonymous": "text",
    "preconceito": "label"
})

df["label"] = df["label"].astype(int)


In [10]:
df.head()

,text,label,num_tokens
0,"Sim, eu sou um ""decepcionado""!\n\nEu vejo pess...",0,147
1,"Em épocas com muitos eventos grandes, altas qu...",0,167
2,Algumas premissas que ele adotou:\n- Ser de di...,0,150
3,Apocalipse 22:11. Cristo fez expiação por Seu ...,0,163
4,:sun_with_face:S:rose:H:cherry_blossom:A:sunfl...,0,103


In [11]:
# Função de pré-processamento
def preprocess_data(df, experiment):
    if 'processed' in experiment:
        print("Pré-processamento ativado.")
        pro_texts = [preprocess(t) for t in df['text']]
    else:
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in df['text']]
    return pro_texts


In [12]:
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "BernoulliNB": BernoulliNB(),
    "MultinomialNB": MultinomialNB(),  # cuidado: exige features não-negativas
    "LinearSVC": LinearSVC(dual="auto", max_iter=10000, random_state=42),
    "KNN": KNeighborsClassifier(),
    "SGDClassifier": SGDClassifier(),
    "RandomForest": RandomForestClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),
    "MLP": MLPClassifier(max_iter=300)
}

In [13]:
df = adicionar_palavras_como_features(df, lexicon)
df.head()

,text,label,num_tokens,aberração,aberração*,aberrações,aleij*,aleija,aleijada,aleijadas,...,progressista,progressistinha,reaça,revolucionário,socialista,sociopata,tucanhalha,vermelhinho,xenofóbico,zumbi
0,"Sim, eu sou um ""decepcionado""!\n\nEu vejo pess...",0,147,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"Em épocas com muitos eventos grandes, altas qu...",0,167,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Algumas premissas que ele adotou:\n- Ser de di...,0,150,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Apocalipse 22:11. Cristo fez expiação por Seu ...,0,163,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,:sun_with_face:S:rose:H:cherry_blossom:A:sunfl...,0,103,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Modelos Pré treinados - Sem pré processamento

In [14]:
# === Pré-processamento dinâmico ===
df["text"] = preprocess_data(df, experiment="raw")  
df.head()

Sem pré-processamento.


,text,label,num_tokens,aberração,aberração*,aberrações,aleij*,aleija,aleijada,aleijadas,...,progressista,progressistinha,reaça,revolucionário,socialista,sociopata,tucanhalha,vermelhinho,xenofóbico,zumbi
0,"sim , eu sou um "" decepcionado "" ! \n\neu vejo...",0,147,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"em épocas com muitos eventos grandes , altas q...",0,167,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,algumas premissas que ele adotou : \n - ser de...,0,150,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,apocalipse 22 : 11 . cristo fez expiação por ...,0,163,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,: sun _ with _ face : s : rose : h : cherry _...,0,103,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### melll-uff/bertweetbr

In [15]:

# === Configurações ===

model_name = "melll-uff/bertweetbr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()



'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
    '''
texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


#X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls")
X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_melll_uff_Sempre_processamento.csv', index=False)

Some weights of RobertaModel were not initialized from the model checkpoint at melll-uff/bertweetbr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===


  File "d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,



=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.773000        0.811348     0.712000  0.758282   
1         BernoulliNB       0.568667        0.607687     0.386667  0.472001   
2       MultinomialNB       0.623000        0.661643     0.504000  0.571796   
3           LinearSVC       0.778667        0.803728     0.738000  0.769232   
4                 KNN       0.717333        0.732344     0.686000  0.707825   
5       SGDClassifier       0.749333        0.759732     0.733333  0.745284   
6        RandomForest       0.732667        0.724930     0.751333  0.737322   
7    GradientBoosting       0.777000        0.786306     0.762667  0.773643   
8                 MLP       0.762667        0.759891     0.770000  0.764220   

   AUC_mean  Accuracy_std  Precision_std  Recall_std    F1_std   AUC_std  \
0  0.854211      0.007557       0.016032    0.009

### FpOliveira/tupi-bert-base-portuguese-cased

In [16]:
# === Configurações ===

model_name = "FpOliveira/tupi-bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''
texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time


print(results_df)
results_df.to_csv('./resultados/results_FpOliveira_Sempre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.813333        0.824631     0.796667  0.810094   
1         BernoulliNB       0.731667        0.763574     0.671333  0.714408   
2       MultinomialNB       0.734000        0.760820     0.682667  0.719602   
3           LinearSVC       0.793000        0.796033     0.789333  0.792292   
4                 KNN       0.766000        0.777279     0.746000  0.761214   
5       SGDClassifier       0.780333        0.791321     0.776667  0.779340   
6        RandomForest       0.774333        0.773096     0.776667  0.774829   
7    GradientBoosting       0.809667        0.810954     0.808667  0.809561   
8                 MLP       0.798333        0.794988     0.804000  0.799427   

   AUC_mean  Accuracy

### ruanchaves/bert-base-portuguese-cased-hatebr

In [17]:
# === Configurações ===

model_name = "ruanchaves/bert-base-portuguese-cased-hatebr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_ruanChaves_Sempre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.807667        0.807439     0.808667  0.807851   
1         BernoulliNB       0.750667        0.724446     0.810667  0.764748   
2       MultinomialNB       0.750667        0.723665     0.812667  0.765197   
3           LinearSVC       0.800667        0.802371     0.798000  0.800132   
4                 KNN       0.761333        0.754348     0.775333  0.764482   
5       SGDClassifier       0.762000        0.790269     0.722000  0.749893   
6        RandomForest       0.784000        0.782770     0.788667  0.785109   
7    GradientBoosting       0.812000        0.812901     0.812667  0.812292   
8                 MLP       0.782667        0.773955     0.800667  0.786612   

   AUC_mean  Accuracy

## Modelo pré treinados - Com pré processamento

In [18]:
# === Pré-processamento dinâmico ===
df["text"] = preprocess_data(df, experiment="processed")  
df.head()

Pré-processamento ativado.


,text,label,num_tokens,aberração,aberração*,aberrações,aleij*,aleija,aleijada,aleijadas,...,progressista,progressistinha,reaça,revolucionário,socialista,sociopata,tucanhalha,vermelhinho,xenofóbico,zumbi
0,sim decepcionar \n\neu vejo pessoa tentar je...,0,147,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,em época muito evento grande alta quantidade e...,0,167,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,algum premissa adotar \n direita \n ter rede s...,0,150,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,apocalipse 22 11 cristo fazer expiação povo a...,0,163,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,sun with face s rose h cherry blossom sunflo...,0,103,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### melll-uff/bertweetbr

In [19]:

# === Configurações ===

model_name = "melll-uff/bertweetbr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''
texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_melll_uff_Compre_processamento.csv', index=False)

Some weights of RobertaModel were not initialized from the model checkpoint at melll-uff/bertweetbr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.776333        0.813813     0.717333  0.762377   
1         BernoulliNB       0.589333        0.597147     0.552000  0.573464   
2       MultinomialNB       0.630000        0.635865     0.610000  0.622433   
3           LinearSVC       0.768667        0.793282     0.726667  0.758504   
4                 KNN       0.710333        0.718920     0.694000  0.705483   
5       SGDClassifier       0.747333        0.750073     0.748000  0.747095   
6        RandomForest       0.721333        0.726168     0.711333  0.718340   
7    GradientBoosting       0.770333        0.782363     0.752000  0.765970   
8                 MLP       0.761667        0.775127     0.738667  0.756189   

   AUC_mean  Accuracy

### FpOliveira/tupi-bert-base-portuguese-cased

In [20]:
# === Configurações ===

model_name = "FpOliveira/tupi-bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    start_model = time.time()  # inicia contador do modelo
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_FpOliveira_Compre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.796667        0.814659     0.768000  0.790591   
1         BernoulliNB       0.686667        0.743219     0.571333  0.645638   
2       MultinomialNB       0.697333        0.745161     0.600667  0.664697   
3           LinearSVC       0.777000        0.786474     0.760667  0.773205   
4                 KNN       0.747333        0.768575     0.708000  0.736805   
5       SGDClassifier       0.771000        0.805873     0.726000  0.758726   
6        RandomForest       0.732667        0.736880     0.724000  0.730150   
7    GradientBoosting       0.795333        0.797942     0.791333  0.794421   
8                 MLP       0.780333        0.784896     0.772667  0.778551   

   AUC_mean  Accuracy

### ruanchaves/bert-base-portuguese-cased-hatebr

In [21]:
# === Configurações ===

model_name = "ruanchaves/bert-base-portuguese-cased-hatebr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''
texts = df["text"].tolist()
labels = df["label"].values


# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_ruanChaves_Compre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.796333        0.801152     0.788667  0.794798   
1         BernoulliNB       0.710667        0.677362     0.806000  0.735831   
2       MultinomialNB       0.711333        0.676648     0.810667  0.737355   
3           LinearSVC       0.790667        0.793181     0.786667  0.789888   
4                 KNN       0.750333        0.745057     0.762000  0.753022   
5       SGDClassifier       0.750333        0.721307     0.820667  0.766131   
6        RandomForest       0.762000        0.779545     0.732667  0.754895   
7    GradientBoosting       0.791667        0.793224     0.790000  0.791301   
8                 MLP       0.781667        0.776510     0.792667  0.784041   

   AUC_mean  Accuracy